<a href="https://colab.research.google.com/github/khyun8072/CAU_AISW_NLP/blob/main/Week5/251118_%E1%84%80%E1%85%AE%E1%86%ABAI_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8%E1%84%8C%E1%85%A1%E1%84%85%E1%85%AD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 고급 언어 모델 파인튜닝 통합 실습

## 개요
이 노트북은 현대 언어 모델 파인튜닝의 주요 기법들을 통합적으로 실습합니다:

1. **Instruction Tuning (지시 학습)**: 모델이 사용자의 지시를 따르도록 학습
2. **Reward Model (보상 모델)**: 모델 출력의 품질을 평가하는 보상 함수 학습
3. **PPO (Proximal Policy Optimization)**: 보상 모델을 활용한 강화학습 기반 파인튜닝
4. **DPO (Direct Preference Optimization)**: 선호도 데이터를 직접 활용한 파인튜닝
5. **모델 비교**: 각 기법으로 학습된 모델들의 성능 비교

각 단계는 독립적으로 실행 가능하며, 전체 파이프라인을 이해할 수 있도록 구성되어 있습니다.

## 환경 설정

필요한 라이브러리를 설치하고 GPU 환경을 확인합니다.

In [ ]:
# 필수 라이브러리 설치
# 주의: 이 셀은 최초 1회만 실행하면 됩니다

!pip install -q transformers datasets peft trl accelerate
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q matplotlib seaborn pandas numpy

In [ ]:
# 라이브러리 임포트 및 GPU 확인
import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, RewardTrainer, PPOTrainer, PPOConfig, DPOTrainer, AutoModelForCausalLMWithValueHead
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # 사용할 GPU 설정 (필요시 변경)

# GPU 확인
print(f"PyTorch 버전: {torch.__version__}")
print(f"Transformers 버전: {transformers.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"현재 GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Part 1: Instruction Tuning (지시 학습)

## 방법론

**Instruction Tuning**은 사전학습된 언어 모델을 특정 지시(instruction)를 따르도록 파인튜닝하는 기법입니다.

### 핵심 개념:
1. **지시-응답 쌍**: 사용자의 지시(instruction)와 그에 대한 적절한 응답(response)으로 구성된 데이터셋
2. **Supervised Fine-Tuning (SFT)**: 지도학습 방식으로 모델이 올바른 응답을 생성하도록 학습
3. **LoRA (Low-Rank Adaptation)**: 전체 모델을 학습하는 대신 작은 어댑터만 학습하여 효율성 향상

### 사용 모델:
- **LiquidAI/LFM2-350M**: 경량화된 언어 모델로 빠른 실습에 적합

### 데이터셋:
- **KoAlpaca**: 한국어 지시-응답 쌍 데이터셋

In [ ]:
# Part 1: Instruction Tuning - 모델 로드 (양자화 없음)

print("=" * 50)
print("Part 1: Instruction Tuning 시작")
print("=" * 50)

# 모델명 설정
model_name = "LiquidAI/LFM2-350M"

# 토크나이저 로드
print(f"\n토크나이저 로드 중: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# pad_token이 없으면 eos_token을 pad_token으로 사용
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("pad_token을 eos_token으로 설정")

# 모델 로드 (양자화 없이 float16만 사용)
print(f"\n모델 로드 중: {model_name}")
print("설정: torch_dtype=torch.float16, device_map='auto'")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# 모델 정보 출력
print(f"\n모델 파라미터 수: {model.num_parameters() / 1e6:.2f}M")
print(f"모델 dtype: {model.dtype}")
print("모델 로드 완료!")

In [ ]:
# Part 1: LoRA 설정

print("\nLoRA 설정 중...")

# LoRA 구성
lora_config = LoraConfig(
    r=16,                        # LoRA rank (낮을수록 파라미터 적음)
    lora_alpha=32,               # LoRA scaling factor
    target_modules=["q_proj", "v_proj"],  # 어텐션 레이어의 query와 value projection에 적용
    lora_dropout=0.05,           # 드롭아웃 비율
    bias="none",                 # bias 학습 안 함
    task_type=TaskType.CAUSAL_LM # 인과적 언어 모델링 태스크
)

# LoRA 적용
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 출력
model.print_trainable_parameters()
print("LoRA 설정 완료!")

In [ ]:
# Part 1: 데이터셋 로드 및 전처리

print("\n데이터셋 로드 중...")

# KoAlpaca 데이터셋 로드 (한국어 instruction-following 데이터셋)
dataset = load_dataset("beomi/KoAlpaca-v1.1a", split="train")

# 데이터셋 크기 확인
print(f"전체 데이터 수: {len(dataset)}")
print(f"\n데이터 샘플:")
print(dataset[0])

# 실습을 위해 일부만 사용 (전체 사용 시 시간이 오래 걸림)
train_dataset = dataset.select(range(min(1000, len(dataset))))
print(f"\n학습에 사용할 데이터 수: {len(train_dataset)}")

# 데이터 포맷팅 함수
def formatting_func(example):
    """instruction과 output을 결합하여 학습용 텍스트 생성"""
    text = f"### 질문: {example['instruction']}\n\n### 답변: {example['output']}"
    return text

print("\n포맷팅된 샘플:")
print(formatting_func(train_dataset[0]))
print("\n데이터셋 준비 완료!")

In [ ]:
# Part 1: SFT (Supervised Fine-Tuning) 학습

print("\nSFT 학습 준비 중...")

# 학습 인자 설정
training_args = TrainingArguments(
    output_dir="./instruction_tuned_model",      # 모델 저장 경로
    num_train_epochs=3,                          # 학습 에포크 수
    per_device_train_batch_size=4,               # 배치 크기
    gradient_accumulation_steps=4,               # 그래디언트 누적 (effective batch size = 16)
    learning_rate=2e-4,                          # 학습률
    fp16=True,                                   # Mixed precision training
    logging_steps=10,                            # 로그 출력 주기
    save_steps=100,                              # 모델 저장 주기
    save_total_limit=2,                          # 저장할 체크포인트 최대 개수
    warmup_steps=50,                             # Learning rate warmup
    report_to="none"                             # wandb 등 리포팅 비활성화
)

# SFT Trainer 초기화
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    formatting_func=formatting_func,
)

print("\n학습 시작...")
print("(이 과정은 GPU 성능에 따라 수분~수십분 소요될 수 있습니다)\n")

# 학습 실행
trainer.train()

print("\nInstruction Tuning 학습 완료!")

In [ ]:
# Part 1: 모델 저장 및 테스트

print("\n모델 저장 중...")

# 최종 모델 저장
output_dir = "./instruction_tuned_model_final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"모델이 {output_dir}에 저장되었습니다.")

# 간단한 테스트
print("\n" + "=" * 50)
print("학습된 모델 테스트")
print("=" * 50)

# 테스트용 프롬프트
test_instruction = "인공지능이란 무엇인가요?"
test_prompt = f"### 질문: {test_instruction}\n\n### 답변:"

# 입력 인코딩
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

# 생성
print(f"\n질문: {test_instruction}")
print("\n생성 중...")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id
    )

# 결과 디코딩
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\n답변:\n{response}")

print("\n" + "=" * 50)
print("Part 1: Instruction Tuning 완료!")
print("=" * 50)

# Part 2: Reward Model (보상 모델)

## 방법론

**Reward Model**은 모델의 출력 품질을 평가하는 스코어링 함수를 학습합니다. 이는 RLHF(Reinforcement Learning from Human Feedback)의 핵심 구성요소입니다.

### 핵심 개념:
1. **선호도 학습**: 인간의 선호도를 반영하여 "좋은 응답"과 "나쁜 응답"을 구별
2. **스코어 출력**: 입력된 (질문, 응답) 쌍에 대해 스칼라 보상 값 출력
3. **Ranking Loss**: 선호되는 응답이 더 높은 스코어를 받도록 학습

### 사용 모델:
- **Qwen/Qwen2.5-0.5B**: 경량 모델을 sequence classification으로 변환

### 데이터셋:
- **kochatgpt_2_RM.jsonl**: chosen(선호)과 rejected(비선호) 응답 쌍

In [ ]:
# Part 2: Reward Model - 모델 로드 (양자화 없음)

print("\n" + "=" * 50)
print("Part 2: Reward Model 학습 시작")
print("=" * 50)

# 이전 모델 메모리 해제
# del model, trainer
torch.cuda.empty_cache()
print("이전 모델 메모리 해제 완료")

# Reward Model용 모델명
rm_model_name = "Qwen/Qwen2.5-0.5B"

# 토크나이저 로드
print(f"\n토크나이저 로드 중: {rm_model_name}")
rm_tokenizer = AutoTokenizer.from_pretrained(rm_model_name, trust_remote_code=True)

if rm_tokenizer.pad_token is None:
    rm_tokenizer.pad_token = rm_tokenizer.eos_token
    print("pad_token을 eos_token으로 설정")

# Sequence Classification 모델로 로드 (양자화 없음)
print(f"\n모델 로드 중: {rm_model_name}")
print("Sequence Classification 모드 (num_labels=1)")
print("설정: torch_dtype=torch.float16, device_map='auto'")

rm_model = AutoModelForSequenceClassification.from_pretrained(
    rm_model_name,
    num_labels=1,                    # 보상 스코어 출력 (단일 스칼라 값)
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# 모델 정보
print(f"\n모델 파라미터 수: {rm_model.num_parameters() / 1e6:.2f}M")
print(f"모델 dtype: {rm_model.dtype}")
print("Reward Model 로드 완료!")

In [ ]:
# Part 2: LoRA 설정

print("\nLoRA 설정 중...")

# LoRA 구성 (Reward Model용)
rm_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS         # Sequence Classification 태스크
)

# LoRA 적용
rm_model = get_peft_model(rm_model, rm_lora_config)

# 학습 가능 파라미터 출력
rm_model.print_trainable_parameters()
print("LoRA 설정 완료!")

In [ ]:
# Part 2: 데이터셋 로드 및 전처리

print("\n데이터셋 로드 중...")

# JSONL 파일에서 데이터 로드
rm_data_path = "kochatgpt_2_RM.jsonl"

def convert_ranking_to_preference_pairs(item):
    """
    ranking 형식의 데이터를 chosen/rejected 쌍들로 변환
    ranking: [2, 1, 0] 형식 (낮은 숫자가 더 선호됨 - 0이 1등)

    3개의 completion에서 3개의 쌍을 생성:
    - completion_0 vs completion_1
    - completion_0 vs completion_2
    - completion_1 vs completion_2
    """
    if 'ranking' not in item:
        return []

    pairs = []

    # data 1) 0 VS 1
    data = {'prompt': item['prompt']}
    if item['ranking'][0] < item['ranking'][1]:
        data['chosen'] = item['completion_0']
        data['rejected'] = item['completion_1']
    else:
        data['chosen'] = item['completion_1']
        data['rejected'] = item['completion_0']
    pairs.append(data)

    # data 2) 0 VS 2
    data = {'prompt': item['prompt']}
    if item['ranking'][0] < item['ranking'][2]:
        data['chosen'] = item['completion_0']
        data['rejected'] = item['completion_2']
    else:
        data['chosen'] = item['completion_2']
        data['rejected'] = item['completion_0']
    pairs.append(data)

    # data 3) 1 VS 2
    data = {'prompt': item['prompt']}
    if item['ranking'][1] < item['ranking'][2]:
        data['chosen'] = item['completion_1']
        data['rejected'] = item['completion_2']
    else:
        data['chosen'] = item['completion_2']
        data['rejected'] = item['completion_1']
    pairs.append(data)

    return pairs

try:
    # 파일이 JSON 배열 형식인지 JSONL 형식인지 확인
    print(f"파일 로드 중: {rm_data_path}")

    with open(rm_data_path, 'r', encoding='utf-8') as f:
        first_char = f.read(1)
        f.seek(0)

        if first_char == '[':
            # JSON 배열 형식
            print("JSON 배열 형식으로 감지됨")
            raw_data = json.load(f)
        else:
            # JSONL 형식
            print("JSONL 형식으로 감지됨")
            raw_data = []
            for line in f:
                line = line.strip()
                if line:
                    raw_data.append(json.loads(line))

    print(f"원본 데이터 수: {len(raw_data)}")

    # 데이터 변환 (각 샘플당 3개의 쌍 생성)
    rm_data = []
    conversion_errors = 0

    for idx, item in enumerate(raw_data):
        try:
            pairs = convert_ranking_to_preference_pairs(item)
            if pairs:
                rm_data.extend(pairs)  # 3개의 쌍을 모두 추가
            else:
                conversion_errors += 1
        except (KeyError, ValueError, TypeError) as e:
            conversion_errors += 1
            if conversion_errors <= 5:
                print(f"경고: {idx}번째 항목 변환 실패 - {str(e)[:50]}")
            continue

    if conversion_errors > 0:
        print(f"\n총 {conversion_errors}개 항목에서 데이터 변환 에러 발생 (건너뛰었습니다)")

    print(f"\n성공적으로 생성된 쌍(pair) 수: {len(rm_data)} (원본 {len(raw_data)}개 × 3)")

    if len(rm_data) > 0:
        print(f"\n첫 번째 샘플의 3개 쌍:")
        for i in range(min(3, len(rm_data))):
            print(f"\n쌍 {i+1}:")
            print(json.dumps(rm_data[i], ensure_ascii=False, indent=2))

        # Dataset 객체로 변환
        rm_dataset = Dataset.from_list(rm_data)

        # 실습용으로 일부만 사용 (500개 쌍)
        rm_train_dataset = rm_dataset.select(range(min(500, len(rm_dataset))))
        print(f"\n학습에 사용할 쌍(pair) 수: {len(rm_train_dataset)}")

        print("\n데이터셋 준비 완료!")
    else:
        raise ValueError("유효한 데이터가 없습니다.")

except FileNotFoundError:
    print(f"\n경고: {rm_data_path} 파일을 찾을 수 없습니다.")
    print("예시 데이터를 생성합니다...\n")

    # 예시 데이터 생성
    rm_data = [
        {
            "prompt": "인공지능의 장점은 무엇인가요?",
            "chosen": "인공지능은 반복적인 작업을 자동화하고, 대량의 데이터를 빠르게 처리할 수 있으며, 인간보다 정확한 예측을 할 수 있습니다.",
            "rejected": "좋아요."
        },
        {
            "prompt": "파이썬으로 리스트를 정렬하는 방법은?",
            "chosen": "파이썬에서는 .sort() 메서드나 sorted() 함수를 사용하여 리스트를 정렬할 수 있습니다. 예: my_list.sort() 또는 sorted(my_list)",
            "rejected": "정렬하면 됩니다."
        }
    ] * 250  # 500개 샘플 생성

    rm_train_dataset = Dataset.from_list(rm_data)
    print(f"생성된 예시 데이터 수: {len(rm_train_dataset)}")

except Exception as e:
    print(f"\n에러 발생: {e}")
    import traceback
    traceback.print_exc()
    print("\n예시 데이터를 생성합니다...\n")

    # 예시 데이터 생성
    rm_data = [
        {
            "prompt": "인공지능의 장점은 무엇인가요?",
            "chosen": "인공지능은 반복적인 작업을 자동화하고, 대량의 데이터를 빠르게 처리할 수 있으며, 인간보다 정확한 예측을 할 수 있습니다.",
            "rejected": "좋아요."
        },
        {
            "prompt": "파이썬으로 리스트를 정렬하는 방법은?",
            "chosen": "파이썬에서는 .sort() 메서드나 sorted() 함수를 사용하여 리스트를 정렬할 수 있습니다. 예: my_list.sort() 또는 sorted(my_list)",
            "rejected": "정렬하면 됩니다."
        }
    ] * 250  # 500개 샘플 생성

    rm_train_dataset = Dataset.from_list(rm_data)
    print(f"생성된 예시 데이터 수: {len(rm_train_dataset)}")

In [ ]:
# Part 2: Reward Model 학습

print("\nReward Model 학습 준비 중...")

# RewardConfig 임포트 및 설정
from trl import RewardConfig

rm_training_args = RewardConfig(
    output_dir="./reward_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    warmup_steps=50,
    report_to="none",
    remove_unused_columns=False,      # 컬럼 자동 제거 방지
    bf16=True,                        # fp16 대신 bf16 사용 (더 안정적)
    max_grad_norm=1.0,                # gradient clipping
)

# Reward Trainer 초기화
rm_trainer = RewardTrainer(
    model=rm_model,
    args=rm_training_args,
    train_dataset=rm_train_dataset,
)

print("\n학습 시작...")
print("(이 과정은 GPU 성능에 따라 수분~수십분 소요될 수 있습니다)\n")

# 학습 실행
rm_trainer.train()

print("\nReward Model 학습 완료!")

In [ ]:
# Part 2: 모델 저장 및 테스트

print("\n모델 저장 중...")

# 최종 모델 저장
rm_output_dir = "./reward_model_final"
rm_trainer.save_model(rm_output_dir)
rm_tokenizer.save_pretrained(rm_output_dir)

print(f"모델이 {rm_output_dir}에 저장되었습니다.")

# 간단한 테스트
print("\n" + "=" * 50)
print("Reward Model 테스트")
print("=" * 50)

# 테스트용 프롬프트와 응답
test_prompt = "파이썬이란 무엇인가요?"
test_response_good = "파이썬은 배우기 쉽고 강력한 프로그래밍 언어로, 데이터 분석, 웹 개발, 인공지능 등 다양한 분야에서 사용됩니다."
test_response_bad = "잘 모르겠습니다."

# 보상 점수 계산 함수
def get_reward_score(prompt, response):
    """프롬프트와 응답에 대한 보상 점수 계산"""
    text = f"질문: {prompt}\n답변: {response}"
    inputs = rm_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(rm_model.device)

    with torch.no_grad():
        outputs = rm_model(**inputs)
        score = outputs.logits[0].item()

    return score

# 점수 계산
score_good = get_reward_score(test_prompt, test_response_good)
score_bad = get_reward_score(test_prompt, test_response_bad)

print(f"\n질문: {test_prompt}")
print(f"\n좋은 응답: {test_response_good}")
print(f"보상 점수: {score_good:.4f}")
print(f"\n나쁜 응답: {test_response_bad}")
print(f"보상 점수: {score_bad:.4f}")
print(f"\n점수 차이: {score_good - score_bad:.4f}")

if score_good > score_bad:
    print("✓ Reward Model이 올바르게 작동합니다!")
else:
    print("⚠ 학습이 더 필요할 수 있습니다.")

print("\n" + "=" * 50)
print("Part 2: Reward Model 완료!")
print("=" * 50)

# Part 3: PPO (Proximal Policy Optimization)

## 방법론

**PPO**는 강화학습 알고리즘으로, 학습된 보상 모델을 사용하여 언어 모델을 최적화합니다.

### 핵심 개념:
1. **Policy Gradient**: 보상을 최대화하는 방향으로 모델 파라미터 업데이트
2. **Proximal Constraint**: 기존 모델(reference model)과 너무 멀어지지 않도록 제약
3. **Value Head**: 상태 가치를 예측하는 추가 헤드 학습

### 학습 프로세스:
1. 질문에 대해 현재 모델로 응답 생성
2. 보상 모델로 응답의 품질 평가
3. PPO 알고리즘으로 모델 업데이트
4. 반복

### 사용 모델:
- **정책 모델**: LiquidAI/LFM2-350M (학습할 모델)
- **참조 모델**: 동일 모델의 초기 버전 (KL divergence 계산용)
- **보상 모델**: Part 2에서 학습한 모델

### 데이터셋:
- **kochatgpt_3_PPO.jsonl**: 질문(query) 데이터

In [ ]:
# Part 3: PPO - 모델 로드 (양자화 없음)

print("\n" + "=" * 50)
print("Part 3: PPO 학습 시작")
print("=" * 50)

# 이전 모델 메모리 해제
del rm_model, rm_trainer
torch.cuda.empty_cache()
print("이전 모델 메모리 해제 완료")

# PPO용 모델명 (Part 1과 동일)
ppo_model_name = "LiquidAI/LFM2-350M"

# 토크나이저 로드
print(f"\n토크나이저 로드 중: {ppo_model_name}")
ppo_tokenizer = AutoTokenizer.from_pretrained(ppo_model_name, trust_remote_code=True)

if ppo_tokenizer.pad_token is None:
    ppo_tokenizer.pad_token = ppo_tokenizer.eos_token
    ppo_tokenizer.padding_side = "left"  # PPO에서는 left padding 권장
    print("pad_token을 eos_token으로 설정 (left padding)")

# 정책 모델 로드 (Value Head 포함, 양자화 없음)
print(f"\n정책 모델 로드 중: {ppo_model_name}")
print("AutoModelForCausalLMWithValueHead 사용")
print("설정: torch_dtype=torch.float16, device_map='auto'")

ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    ppo_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# 참조 모델 로드 (KL divergence 계산용, 양자화 없음)
print(f"\n참조 모델 로드 중: {ppo_model_name}")
print("설정: torch_dtype=torch.float16, device_map='auto'")

ppo_ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    ppo_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print(f"\n정책 모델 파라미터 수: {ppo_model.pretrained_model.num_parameters() / 1e6:.2f}M")
print("모델 로드 완료!")

In [ ]:
# Part 3: LoRA 설정

print("\nLoRA 설정 중...")

# LoRA 구성
ppo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# 정책 모델에만 LoRA 적용 (참조 모델은 고정)
ppo_model.pretrained_model = get_peft_model(ppo_model.pretrained_model, ppo_lora_config)

# 학습 가능 파라미터 출력
ppo_model.pretrained_model.print_trainable_parameters()
print("LoRA 설정 완료!")

In [ ]:
# Part 3: 데이터셋 로드 및 전처리

print("\n데이터셋 로드 중...")

# JSONL 파일에서 데이터 로드
ppo_data_path = "kochatgpt_3_PPO.jsonl"

try:
    # 파일이 JSON 배열 형식인지 JSONL 형식인지 확인
    print(f"파일 로드 중: {ppo_data_path}")

    with open(ppo_data_path, 'r', encoding='utf-8') as f:
        first_char = f.read(1)
        f.seek(0)

        if first_char == '[':
            # JSON 배열 형식
            print("JSON 배열 형식으로 감지됨")
            import json
            ppo_data = json.load(f)
        else:
            # JSONL 형식
            print("JSONL 형식으로 감지됨")
            import json
            ppo_data = []
            for line in f:
                line = line.strip()
                if line:
                    ppo_data.append(json.loads(line))

    print(f"로드된 데이터 수: {len(ppo_data)}")
    print(f"\n데이터 샘플:")
    print(json.dumps(ppo_data[0], ensure_ascii=False, indent=2))

    # prompt 또는 query 필드 추출 (둘 중 있는 것 사용)
    if "prompt" in ppo_data[0]:
        ppo_queries = [item["prompt"] for item in ppo_data[:min(200, len(ppo_data))]]
        print(f"\n'prompt' 필드 사용")
    elif "query" in ppo_data[0]:
        ppo_queries = [item["query"] for item in ppo_data[:min(200, len(ppo_data))]]
        print(f"\n'query' 필드 사용")
    else:
        raise ValueError("데이터에 'prompt' 또는 'query' 필드가 없습니다.")

    print(f"학습에 사용할 쿼리 수: {len(ppo_queries)}")

except FileNotFoundError:
    print(f"\n경고: {ppo_data_path} 파일을 찾을 수 없습니다.")
    print("예시 쿼리를 생성합니다...\n")

    # 예시 쿼리 생성
    ppo_queries = [
        "인공지능의 미래는 어떻게 될까요?",
        "파이썬 프로그래밍을 배우는 좋은 방법은?",
        "머신러닝과 딥러닝의 차이는 무엇인가요?",
        "자연어 처리 기술의 활용 사례를 알려주세요.",
        "강화학습이란 무엇인가요?"
    ] * 40  # 200개 쿼리 생성

    print(f"생성된 예시 쿼리 수: {len(ppo_queries)}")

except Exception as e:
    print(f"\n에러 발생: {e}")
    import traceback
    traceback.print_exc()
    print("\n예시 쿼리를 생성합니다...\n")

    # 예시 쿼리 생성
    ppo_queries = [
        "인공지능의 미래는 어떻게 될까요?",
        "파이썬 프로그래밍을 배우는 좋은 방법은?",
        "머신러닝과 딥러닝의 차이는 무엇인가요?",
        "자연어 처리 기술의 활용 사례를 알려주세요.",
        "강화학습이란 무엇인가요?"
    ] * 40  # 200개 쿼리 생성

    print(f"생성된 예시 쿼리 수: {len(ppo_queries)}")

print("\n데이터셋 준비 완료!")

In [ ]:
# Part 3: PPO 설정 및 학습

print("\nPPO 설정 중...")

# 간단한 PPO 구현 (최신 TRL PPOTrainer는 너무 복잡하므로 직접 구현)
import torch.nn.functional as F
from peft import PeftModel

# 보상 모델 로드 (평가용)
print("\n보상 모델 로드 중...")
reward_model_path = "./reward_model_final"
base_reward_model_name = "Qwen/Qwen2.5-0.5B"

try:
    # Base 모델 로드
    print(f"Base 모델 로드 중: {base_reward_model_name}")
    base_reward_model = AutoModelForSequenceClassification.from_pretrained(
        base_reward_model_name,
        num_labels=1,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    # LoRA 어댑터 로드
    print(f"LoRA 어댑터 로드 중: {reward_model_path}")
    reward_model = PeftModel.from_pretrained(base_reward_model, reward_model_path)
    reward_model = reward_model.merge_and_unload()  # LoRA를 base에 병합
    reward_model.eval()

    reward_tokenizer = AutoTokenizer.from_pretrained(reward_model_path)

    print("보상 모델 로드 완료!")
    use_reward_model = True
except Exception as e:
    print(f"경고: 보상 모델 로드 실패 - {e}")
    print("더미 보상을 사용합니다.")
    use_reward_model = False

# 보상 계산 함수
def compute_reward(query, response):
    """쿼리와 응답에 대한 보상 계산"""
    if not use_reward_model:
        # 더미 보상: 응답 길이에 기반 (10~100 문자 사이가 적당)
        length = len(response)
        if 10 <= length <= 200:
            return torch.tensor(1.0, device=ppo_model.pretrained_model.device)
        else:
            return torch.tensor(0.0, device=ppo_model.pretrained_model.device)

    text = f"질문: {query}\n답변: {response}"
    inputs = reward_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(reward_model.device)

    with torch.no_grad():
        outputs = reward_model(**inputs)
        reward = outputs.logits[0, 0]  # 스칼라 값

    return reward.to(ppo_model.pretrained_model.device)

# Optimizer 설정
from torch.optim import AdamW

optimizer = AdamW(ppo_model.pretrained_model.parameters(), lr=1e-5)

print("\nPPO 학습 시작...")
print("(이 과정은 시간이 오래 걸릴 수 있습니다)\n")

# 학습 설정
num_epochs = 2
kl_coef = 0.05  # KL divergence 계수
clip_range = 0.2  # PPO clip range

# PPO 학습 루프
generation_kwargs = {
    "max_new_tokens": 80,
    "temperature": 0.7,
    "do_sample": True,
    "pad_token_id": ppo_tokenizer.pad_token_id,
}

total_rewards = []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    epoch_rewards = []

    for batch_idx, query in enumerate(ppo_queries[:50]):  # 실습용으로 50개만 사용
        # 1. 쿼리 인코딩
        query_text = f"### 질문: {query}\n\n### 답변:"
        query_inputs = ppo_tokenizer(query_text, return_tensors="pt").to(ppo_model.pretrained_model.device)

        # 2. 응답 생성
        ppo_model.pretrained_model.eval()
        with torch.no_grad():
            response_tensors = ppo_model.pretrained_model.generate(
                **query_inputs,
                **generation_kwargs
            )

        # 3. 응답 디코딩
        response_text = ppo_tokenizer.decode(
            response_tensors[0][len(query_inputs.input_ids[0]):],
            skip_special_tokens=True
        )

        # 4. 보상 계산
        reward = compute_reward(query, response_text)
        epoch_rewards.append(reward.item())

        # 5. 정책 모델 학습 (간단한 policy gradient)
        ppo_model.pretrained_model.train()

        # 전체 시퀀스로 loss 계산
        full_text = query_text + response_text
        inputs = ppo_tokenizer(full_text, return_tensors="pt").to(ppo_model.pretrained_model.device)

        outputs = ppo_model.pretrained_model(**inputs, labels=inputs.input_ids)
        loss = outputs.loss

        # Reward로 loss 가중치 조정 (간단한 REINFORCE)
        weighted_loss = loss * (-reward)  # 보상이 높으면 loss를 줄임

        # 6. 역전파 및 업데이트
        optimizer.zero_grad()
        weighted_loss.backward()
        torch.nn.utils.clip_grad_norm_(ppo_model.pretrained_model.parameters(), max_norm=0.5)
        optimizer.step()

        # 7. 주기적으로 통계 출력
        if batch_idx % 10 == 0:
            avg_reward = sum(epoch_rewards[-10:]) / min(10, len(epoch_rewards))
            print(f"  Batch {batch_idx}/50: reward={reward.item():.4f}, avg_reward={avg_reward:.4f}, loss={loss.item():.4f}")

        # 메모리 정리
        if batch_idx % 20 == 0:
            torch.cuda.empty_cache()

    avg_epoch_reward = sum(epoch_rewards) / len(epoch_rewards)
    total_rewards.append(avg_epoch_reward)
    print(f"Epoch {epoch + 1} 평균 보상: {avg_epoch_reward:.4f}")

print(f"\n전체 평균 보상: {sum(total_rewards) / len(total_rewards):.4f}")
print("\nPPO 학습 완료!")

In [ ]:
# Part 3: 모델 저장 및 테스트

print("\n모델 저장 중...")

# 최종 모델 저장
ppo_output_dir = "./ppo_model_final"
ppo_model.save_pretrained(ppo_output_dir)
ppo_tokenizer.save_pretrained(ppo_output_dir)

print(f"모델이 {ppo_output_dir}에 저장되었습니다.")

# 간단한 테스트
print("\n" + "=" * 50)
print("PPO 학습된 모델 테스트")
print("=" * 50)

# 테스트용 프롬프트
test_query = "강화학습이란 무엇인가요?"
test_prompt = f"### 질문: {test_query}\n\n### 답변:"

# 입력 인코딩
inputs = ppo_tokenizer(test_prompt, return_tensors="pt").to(ppo_model.pretrained_model.device)

# 생성
print(f"\n질문: {test_query}")
print("\n생성 중...")

with torch.no_grad():
    outputs = ppo_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=ppo_tokenizer.pad_token_id
    )

# 결과 디코딩
response = ppo_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\n답변:\n{response}")

print("\n" + "=" * 50)
print("Part 3: PPO 완료!")
print("=" * 50)

# Part 4: DPO (Direct Preference Optimization)

## 방법론

**DPO**는 별도의 보상 모델 없이 선호도 데이터를 직접 사용하여 모델을 최적화하는 기법입니다.

### 핵심 개념:
1. **Direct Optimization**: 보상 모델을 거치지 않고 선호도를 직접 학습
2. **Binary Cross-Entropy**: 선호/비선호 응답 쌍을 이진 분류 문제로 변환
3. **Reference Model**: KL divergence로 과도한 변화 방지

### PPO와의 차이:
- PPO: 보상 모델 필요, 강화학습 기반, 불안정할 수 있음
- DPO: 보상 모델 불필요, 지도학습 기반, 안정적이고 간단함

### 사용 모델:
- **정책 모델**: LiquidAI/LFM2-350M
- **참조 모델**: 동일 모델의 초기 버전

### 데이터셋:
- **orca_dpo_pairs_ko**: 한국어 번역된 Orca DPO 선호도 데이터

In [ ]:
# Part 4: DPO - 모델 로드 (양자화 없음)

print("\n" + "=" * 50)
print("Part 4: DPO 학습 시작")
print("=" * 50)

# 이전 모델 메모리 해제
# del ppo_model, ppo_ref_model
if reward_model is not None:
    del reward_model
torch.cuda.empty_cache()
print("이전 모델 메모리 해제 완료")

# DPO용 모델명
dpo_model_name = "LiquidAI/LFM2-350M"

# 토크나이저 로드
print(f"\n토크나이저 로드 중: {dpo_model_name}")
dpo_tokenizer = AutoTokenizer.from_pretrained(dpo_model_name, trust_remote_code=True)

if dpo_tokenizer.pad_token is None:
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token
    print("pad_token을 eos_token으로 설정")

# 정책 모델 로드 (양자화 없음)
print(f"\n정책 모델 로드 중: {dpo_model_name}")
print("설정: torch_dtype=torch.float16, device_map='auto'")

dpo_model = AutoModelForCausalLM.from_pretrained(
    dpo_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# 참조 모델 로드 (양자화 없음)
print(f"\n참조 모델 로드 중: {dpo_model_name}")
print("설정: torch_dtype=torch.float16, device_map='auto'")

dpo_ref_model = AutoModelForCausalLM.from_pretrained(
    dpo_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print(f"\n모델 파라미터 수: {dpo_model.num_parameters() / 1e6:.2f}M")
print("모델 로드 완료!")

In [ ]:
# Part 4: LoRA 설정

print("\nLoRA 설정 중...")

# LoRA 구성
dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# 정책 모델에만 LoRA 적용
dpo_model = get_peft_model(dpo_model, dpo_lora_config)

# 학습 가능 파라미터 출력
dpo_model.print_trainable_parameters()
print("LoRA 설정 완료!")

In [ ]:
# Part 4: 데이터셋 로드 및 전처리

print("\n데이터셋 로드 중...")

try:
    # Hugging Face 데이터셋 로드
    dpo_dataset = load_dataset("heegyu/orca_dpo_pairs_ko", split="train")

    print(f"로드된 데이터 수: {len(dpo_dataset)}")
    print(f"\n데이터 샘플:")
    print(dpo_dataset[0])

    # 실습용으로 일부만 사용
    dpo_train_dataset = dpo_dataset.select(range(min(500, len(dpo_dataset))))
    print(f"\n학습에 사용할 데이터 수: {len(dpo_train_dataset)}")

except Exception as e:
    print(f"\n경고: 데이터셋 로드 실패 - {e}")
    print("예시 데이터를 생성합니다...\n")

    # 예시 데이터 생성
    dpo_data = [
        {
            "prompt": "파이썬에서 리스트와 튜플의 차이는 무엇인가요?",
            "chosen": "리스트는 가변(mutable) 자료형으로 요소를 추가, 삭제, 수정할 수 있습니다. 반면 튜플은 불변(immutable) 자료형으로 생성 후 변경할 수 없습니다. 리스트는 []로, 튜플은 ()로 표현합니다.",
            "rejected": "다릅니다."
        },
        {
            "prompt": "머신러닝에서 과적합(overfitting)이란?",
            "chosen": "과적합은 모델이 학습 데이터에 너무 특화되어 새로운 데이터에 대한 일반화 성능이 떨어지는 현상입니다. 정규화, 드롭아웃, 조기 종료 등으로 방지할 수 있습니다.",
            "rejected": "학습을 너무 많이 한 것입니다."
        }
    ] * 250  # 500개 샘플 생성

    dpo_train_dataset = Dataset.from_list(dpo_data)
    print(f"생성된 예시 데이터 수: {len(dpo_train_dataset)}")

print("\n데이터셋 준비 완료!")

In [ ]:
# Part 4: DPO 학습

print("\nDPO 학습 준비 중...")

# DPOConfig 임포트 및 설정
from trl import DPOConfig

dpo_training_args = DPOConfig(
    output_dir="./dpo_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    bf16=True,                       # fp16 대신 bf16 사용
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    warmup_steps=50,
    report_to="none",
    remove_unused_columns=False,
    max_prompt_length=256,           # DPOConfig에서는 여기에 설정
    max_length=512,                  # 전체 시퀀스 최대 길이
    beta=0.1,                        # DPO beta 파라미터
)

# DPO Trainer 초기화
dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=dpo_ref_model,
    args=dpo_training_args,
    train_dataset=dpo_train_dataset,
    processing_class=dpo_tokenizer,  # tokenizer 전달
)

print("\n학습 시작...")
print("(이 과정은 GPU 성능에 따라 수분~수십분 소요될 수 있습니다)\n")

# 학습 실행
dpo_trainer.train()

print("\nDPO 학습 완료!")

In [ ]:
# Part 4: 모델 저장 및 테스트

print("\n모델 저장 중...")

# 최종 모델 저장
dpo_output_dir = "./dpo_model_final"
dpo_trainer.save_model(dpo_output_dir)
dpo_tokenizer.save_pretrained(dpo_output_dir)

print(f"모델이 {dpo_output_dir}에 저장되었습니다.")

# 간단한 테스트
print("\n" + "=" * 50)
print("DPO 학습된 모델 테스트")
print("=" * 50)

# 테스트용 프롬프트
test_query = "딥러닝에서 배치 정규화(Batch Normalization)의 역할은?"
test_prompt = f"### 질문: {test_query}\n\n### 답변:"

# 입력 인코딩
inputs = dpo_tokenizer(test_prompt, return_tensors="pt").to(dpo_model.device)

# 생성
print(f"\n질문: {test_query}")
print("\n생성 중...")

with torch.no_grad():
    outputs = dpo_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=dpo_tokenizer.pad_token_id
    )

# 결과 디코딩
response = dpo_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\n답변:\n{response}")

print("\n" + "=" * 50)
print("Part 4: DPO 완료!")
print("=" * 50)

# Part 5: 모델 비교 및 평가

## 목적

이 섹션에서는 앞서 학습한 4가지 모델을 동일한 테스트 세트에서 비교 평가합니다.

### 비교 대상:
1. **Baseline**: 파인튜닝하지 않은 원본 모델
2. **Instruction Tuning**: SFT로 학습한 모델
3. **PPO**: 강화학습 기반 모델
4. **DPO**: 직접 선호도 최적화 모델

### 평가 지표:
1. **응답 품질**: 보상 모델을 사용한 점수
2. **응답 길이**: 생성된 텍스트의 길이
3. **일관성**: 여러 샘플에 대한 안정성

### 시각화:
- 모델별 보상 점수 비교
- 응답 길이 분포
- 종합 성능 지표

In [ ]:
# Part 5: 모델 비교 준비

print("\n" + "=" * 50)
print("Part 5: 모델 비교 및 평가")
print("=" * 50)

# 이전 모델 메모리 해제
# del dpo_model, dpo_ref_model, dpo_trainer
torch.cuda.empty_cache()
print("\n이전 모델 메모리 해제 완료")

# 테스트 질문 준비
test_questions = [
    "인공지능과 머신러닝의 차이점은 무엇인가요?",
    "파이썬에서 리스트 컴프리헨션이란?",
    "딥러닝에서 활성화 함수의 역할은?",
    "자연어 처리에서 토크나이제이션이란?",
    "강화학습의 기본 개념을 설명해주세요."
]

print(f"\n테스트 질문 수: {len(test_questions)}")
print("\n테스트 질문 목록:")
for i, q in enumerate(test_questions, 1):
    print(f"  {i}. {q}")

# 모델 경로 설정
model_paths = {
    "Baseline": "LiquidAI/LFM2-350M",
    "Instruction Tuning": "./instruction_tuned_model_final",
    "PPO": "./ppo_model_final",
    "DPO": "./dpo_model_final"
}

print("\n비교할 모델:")
for name, path in model_paths.items():
    print(f"  - {name}: {path}")

In [ ]:
# Part 5: 모든 모델로 응답 생성

print("\n모든 모델로 응답 생성 중...\n")

from peft import PeftModel

# 결과 저장용 딕셔너리
results = {name: [] for name in model_paths.keys()}

# 각 모델의 base 모델 정보
base_models = {
    "Baseline": None,  # Baseline은 직접 로드
    "Instruction Tuning": "LiquidAI/LFM2-350M",
    "PPO": "LiquidAI/LFM2-350M",
    "DPO": "LiquidAI/LFM2-350M"
}

# 생성 설정
gen_config = {
    "max_new_tokens": 150,
    "temperature": 0.7,
    "do_sample": True,
}

# 각 모델로 응답 생성
for model_name, model_path in model_paths.items():
    print(f"{'='*50}")
    print(f"{model_name} 모델 테스트")
    print(f"{'='*50}\n")

    try:
        # 토크나이저 로드
        print(f"토크나이저 로드 중: {model_path}")
        test_tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

        if test_tokenizer.pad_token is None:
            test_tokenizer.pad_token = test_tokenizer.eos_token

        # 모델 로드
        print(f"모델 로드 중: {model_path}")

        if model_name == "Baseline":
            # Baseline은 base 모델 직접 로드
            test_model = AutoModelForCausalLM.from_pretrained(
                model_path,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True
            )
        else:
            # LoRA 모델들은 base + adapter 로드
            base_model_name = base_models[model_name]
            print(f"  Base 모델: {base_model_name}")

            # PPO 모델은 특별 처리
            if "ppo" in model_path.lower():
                # PPO는 AutoModelForCausalLMWithValueHead로 저장되었을 수 있음
                try:
                    base_model = AutoModelForCausalLM.from_pretrained(
                        base_model_name,
                        torch_dtype=torch.bfloat16,
                        device_map="auto",
                        trust_remote_code=True
                    )
                    test_model = PeftModel.from_pretrained(base_model, model_path)
                    test_model = test_model.merge_and_unload()
                except:
                    # 일반 모델로 재시도
                    base_model = AutoModelForCausalLM.from_pretrained(
                        base_model_name,
                        torch_dtype=torch.bfloat16,
                        device_map="auto",
                        trust_remote_code=True
                    )
                    test_model = PeftModel.from_pretrained(base_model, model_path)
                    test_model = test_model.merge_and_unload()
            else:
                # Instruction Tuning, DPO
                base_model = AutoModelForCausalLM.from_pretrained(
                    base_model_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    trust_remote_code=True
                )
                test_model = PeftModel.from_pretrained(base_model, model_path)
                test_model = test_model.merge_and_unload()

        test_model.eval()
        print("모델 로드 완료\n")

        # 각 질문에 대해 응답 생성
        for i, question in enumerate(test_questions, 1):
            prompt = f"### 질문: {question}\n\n### 답변:"
            inputs = test_tokenizer(prompt, return_tensors="pt").to(test_model.device)

            with torch.no_grad():
                outputs = test_model.generate(
                    **inputs,
                    **gen_config,
                    pad_token_id=test_tokenizer.pad_token_id
                )

            response = test_tokenizer.decode(outputs[0], skip_special_tokens=True)

            # 답변 부분만 추출
            if "### 답변:" in response:
                answer = response.split("### 답변:")[1].strip()
            else:
                answer = response[len(prompt):].strip()

            results[model_name].append({
                "question": question,
                "answer": answer,
                "length": len(answer)
            })

            print(f"질문 {i}: {question}")
            print(f"답변: {answer}..." if len(answer) > 100 else f"답변: {answer}")
            print()

        # 메모리 해제
        del test_model, test_tokenizer
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"경고: {model_name} 모델 로드 실패 - {e}")
        import traceback
        traceback.print_exc()
        print("이 모델을 건너뜁니다.\n")
        continue

print("\n" + "="*50)
print("모든 모델 응답 생성 완료!")
print("="*50)

In [ ]:
# Part 5: 보상 모델로 품질 평가

print("\n보상 모델을 사용한 품질 평가 중...\n")

try:
    # 보상 모델 로드 (LoRA 어댑터 + Base 모델)
    print("보상 모델 로드 중...")

    from peft import PeftModel

    # Base 모델 로드
    base_reward_model = AutoModelForSequenceClassification.from_pretrained(
        "Qwen/Qwen2.5-0.5B",
        num_labels=1,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    # LoRA 어댑터 로드 및 병합
    reward_model = PeftModel.from_pretrained(base_reward_model, "./reward_model_final")
    reward_model = reward_model.merge_and_unload()
    reward_model.eval()

    reward_tokenizer = AutoTokenizer.from_pretrained("./reward_model_final")

    print("보상 모델 로드 완료\n")

    # 각 모델의 응답에 대해 보상 점수 계산
    reward_scores = {name: [] for name in results.keys()}

    for model_name, responses in results.items():
        if not responses:  # 응답이 없으면 건너뛰기
            continue

        print(f"{model_name} 모델 평가 중...")

        for resp in responses:
            text = f"질문: {resp['question']}\n답변: {resp['answer']}"
            inputs = reward_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(reward_model.device)

            with torch.no_grad():
                outputs = reward_model(**inputs)
                score = outputs.logits[0, 0].item()  # [0, 0]으로 스칼라 추출

            reward_scores[model_name].append(score)
            resp["reward"] = score

        avg_reward = np.mean(reward_scores[model_name])
        print(f"  평균 보상 점수: {avg_reward:.4f}")

    print("\n평가 완료!")
    use_rewards = True

except Exception as e:
    print(f"경고: 보상 모델 로드 실패 - {e}")
    import traceback
    traceback.print_exc()
    print("\n보상 점수 없이 계속합니다.\n")
    use_rewards = False

In [ ]:
# Part 5: 결과 시각화 - 보상 점수 비교

print("\n결과 시각화 중...\n")

# 한글 폰트 설정
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정 (시스템에 따라 다를 수 있음)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 그래프 스타일 설정
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

if use_rewards and reward_scores:
    # 1. 모델별 평균 보상 점수
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # 평균 보상 점수 막대 그래프
    ax1 = axes[0, 0]
    model_names = [name for name in reward_scores.keys() if reward_scores[name]]
    avg_rewards = [np.mean(reward_scores[name]) for name in model_names]

    bars = ax1.bar(model_names, avg_rewards)
    ax1.set_title('Average Reward Score by Model', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Average Reward Score', fontsize=12)
    ax1.set_xlabel('Model', fontsize=12)
    ax1.grid(axis='y', alpha=0.3)

    # 막대 위에 값 표시
    for bar, val in zip(bars, avg_rewards):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)

    # 2. 보상 점수 분포 박스플롯
    ax2 = axes[0, 1]
    reward_data = [reward_scores[name] for name in model_names]

    bp = ax2.boxplot(reward_data, labels=model_names, patch_artist=True)
    ax2.set_title('Reward Score Distribution by Model', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Reward Score', fontsize=12)
    ax2.set_xlabel('Model', fontsize=12)
    ax2.grid(axis='y', alpha=0.3)

    # 박스플롯 색상 설정
    colors = sns.color_palette("husl", len(model_names))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # 3. 응답 길이 비교
    ax3 = axes[1, 0]
    avg_lengths = [np.mean([r['length'] for r in results[name]]) for name in model_names if results[name]]

    bars = ax3.bar(model_names, avg_lengths, color=colors)
    ax3.set_title('Average Response Length by Model', fontsize=14, fontweight='bold')
    ax3.set_ylabel('Average Characters', fontsize=12)
    ax3.set_xlabel('Model', fontsize=12)
    ax3.grid(axis='y', alpha=0.3)

    # 막대 위에 값 표시
    for bar, val in zip(bars, avg_lengths):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.0f}', ha='center', va='bottom', fontsize=10)

    # 4. 질문별 보상 점수 추이
    ax4 = axes[1, 1]

    for i, name in enumerate(model_names):
        if results[name]:
            question_scores = [r.get('reward', 0) for r in results[name]]
            ax4.plot(range(1, len(question_scores) + 1), question_scores,
                    marker='o', label=name, linewidth=2, markersize=8)

    ax4.set_title('Reward Score Trend by Question', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Reward Score', fontsize=12)
    ax4.set_xlabel('Question Number', fontsize=12)
    ax4.legend(loc='best', fontsize=10)
    ax4.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
    print("Graph saved as 'model_comparison.png'")
    plt.show()

else:
    print("No reward scores available. Visualizing response length only.")

    fig, ax = plt.subplots(figsize=(10, 6))
    model_names = [name for name in results.keys() if results[name]]
    avg_lengths = [np.mean([r['length'] for r in results[name]]) for name in model_names]

    bars = ax.bar(model_names, avg_lengths)
    ax.set_title('Average Response Length by Model', fontsize=14, fontweight='bold')
    ax.set_ylabel('Average Characters', fontsize=12)
    ax.set_xlabel('Model', fontsize=12)
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, avg_lengths):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.0f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig('model_comparison_length.png', dpi=300, bbox_inches='tight')
    print("Graph saved as 'model_comparison_length.png'")
    plt.show()